In [ ]:
import requests
import xarray as xr
import pandas as pd
from pathlib import Path
import rasterio
import numpy as np
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:
# ============================================================
# NOAA GEFSv12 REFORECAST DOWNLOADER
#
# This cell downloads historical GEFSv12 reforecast files and
# builds a forecast dataframe called forecast_df.
#
# forecast_df contains:
#   - forecast start time
#   - forecast valid time
#   - lead time in hours
#   - latitude / longitude
#   - predicted 2m temperature in Celsius
#   - precipitation
#   - cloud cover
# ============================================================


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

START_DATE = "2015-01-01"
END_DATE = "2015-01-02"

# Forecast member:
# c00 = control forecast
MEMBER = "c00"

# Variables from NOAA GEFS reforecast
VARIABLES = {
    "tmp_2m": "predicted_temp_k",   # 2 metre forecast temperature, Kelvin
    "apcp_sfc": "precipitation",    # accumulated precipitation
    "tcdc_eatm": "cloud_cover"      # total cloud cover
}

# Forecast lead times to extract
LEAD_HOURS = [30]

# Local folder for downloaded GRIB2 files
DATA_DIR = Path("gefs_downloads")
DATA_DIR.mkdir(exist_ok=True)


# ------------------------------------------------------------
# Download one NOAA GEFS file
# ------------------------------------------------------------

def download_gefs_file(year, run, variable, member="c00"):
    """
    Downloads one GEFSv12 reforecast GRIB2 file if it does not
    already exist locally.

    Example run:
        2010010100 = forecast initialized at 2010-01-01 00 UTC
    """

    url = (
        f"https://noaa-gefs-retrospective.s3.amazonaws.com/"
        f"GEFSv12/reforecast/{year}/{run}/{member}/Days:1-10/"
        f"{variable}_{run}_{member}.grib2"
    )

    folder = DATA_DIR / str(year) / variable
    folder.mkdir(parents=True, exist_ok=True)

    filename = folder / f"{variable}_{run}_{member}.grib2"

    # Use existing file if it has already been downloaded
    if filename.exists():
        return filename

    print("Downloading:", url)
    response = requests.get(url)

    # If NOAA/AWS does not have this file, skip it
    if response.status_code != 200:
        print("Missing file:", url)
        return None

    with open(filename, "wb") as f:
        f.write(response.content)

    return filename


# ------------------------------------------------------------
# Read one GRIB2 variable into a dataframe
# ------------------------------------------------------------

def read_gefs_variable(filename, variable, new_column_name):
    """
    Reads one GEFS GRIB2 file.

    Steps:
      1. Opens the file with xarray/cfgrib
      2. Reduces the 0.25-degree grid to about 1-degree spacing
      3. Keeps only the USA approximate box
      4. Keeps only selected lead times, e.g. 30h and 60h
      5. Returns a dataframe with a clear column name
    """

    ds = xr.open_dataset(filename, engine="cfgrib")

    # GEFS grid is 0.25 degrees.
    # Taking every 4th point gives about 1-degree spacing.
    ds = ds.isel(
        latitude=slice(0, None, 4),
        longitude=slice(0, None, 4)
    )

    # USA approximate box.
    # GEFS latitude runs north -> south, so slice(50, 25) is correct.
    # GEFS longitude uses 0-360 degrees, so 235-295 is western/central USA.
    ds = ds.sel(
        latitude=slice(50, 25),
        longitude=slice(235, 295)
    )

    rows = []

    for lead in LEAD_HOURS:
        # Select forecast lead time, e.g. 30 hours or 60 hours
        ds_lead = ds.sel(step=pd.Timedelta(hours=lead))

        # Convert xarray dataset to pandas dataframe
        df_var = ds_lead.to_dataframe().reset_index()

        # Store lead time as a normal numeric column
        df_var["lead_hours"] = lead

        rows.append(df_var)

    df_var = pd.concat(rows, ignore_index=True)

    # Get the internal variable name used by cfgrib/xarray.
    # Example: tmp_2m often appears as t2m.
    data_var_name = list(ds.data_vars)[0]

    # Rename the internal variable to our clearer column name
    df_var = df_var.rename(columns={data_var_name: new_column_name})

    # Keep only columns needed for joining variables together
    df_var = df_var[
        [
            "time",
            "valid_time",
            "lead_hours",
            "latitude",
            "longitude",
            new_column_name
        ]
    ]

    return df_var


# ------------------------------------------------------------
# Main loop: build forecast_df
# ------------------------------------------------------------

all_days = []
dates = pd.date_range(START_DATE, END_DATE, freq="D")

for date in dates:

    year = date.year
    run = date.strftime("%Y%m%d") + "00"

    print("Processing:", run)

    daily_dfs = []

    for variable, new_column_name in VARIABLES.items():

        filename = download_gefs_file(
            year=year,
            run=run,
            variable=variable,
            member=MEMBER
        )

        if filename is None:
            continue

        df_var = read_gefs_variable(
            filename=filename,
            variable=variable,
            new_column_name=new_column_name
        )

        # Delete GRIB2 file after extracting useful data.
        # The .idx file may remain; that is fine.
        try:
            filename.unlink()
        except PermissionError:
            print("Could not delete file:", filename)

        daily_dfs.append(df_var)

    # Only use this day if all requested variables were found
    if len(daily_dfs) == len(VARIABLES):

        df_day = daily_dfs[0]

        # Merge temperature, precipitation, and cloud cover
        for next_df in daily_dfs[1:]:
            df_day = df_day.merge(
                next_df,
                on=[
                    "time",
                    "valid_time",
                    "lead_hours",
                    "latitude",
                    "longitude"
                ],
                how="inner"
            )

        df_day["year"] = year
        df_day["run"] = run

        all_days.append(df_day)

# Combine all forecast days into one dataframe
forecast_df = pd.concat(all_days, ignore_index=True)

# Convert forecast temperature from Kelvin to Celsius
forecast_df["predicted_temp_c"] = forecast_df["predicted_temp_k"] - 273.15

# Drop Kelvin column after conversion
forecast_df = forecast_df.drop(columns=["predicted_temp_k"])

# Make sure time columns are datetime
forecast_df["time"] = pd.to_datetime(forecast_df["time"])
forecast_df["valid_time"] = pd.to_datetime(forecast_df["valid_time"])

print(forecast_df.head())
print(forecast_df.shape)



Processing: 2015010100
Downloading: https://noaa-gefs-retrospective.s3.amazonaws.com/GEFSv12/reforecast/2015/2015010100/c00/Days:1-10/tmp_2m_2015010100_c00.grib2


Ignoring index file 'gefs_downloads\\2015\\tmp_2m\\tmp_2m_2015010100_c00.grib2.5b7b6.idx' older than GRIB file


Downloading: https://noaa-gefs-retrospective.s3.amazonaws.com/GEFSv12/reforecast/2015/2015010100/c00/Days:1-10/apcp_sfc_2015010100_c00.grib2


Ignoring index file 'gefs_downloads\\2015\\apcp_sfc\\apcp_sfc_2015010100_c00.grib2.5b7b6.idx' older than GRIB file


Downloading: https://noaa-gefs-retrospective.s3.amazonaws.com/GEFSv12/reforecast/2015/2015010100/c00/Days:1-10/tcdc_eatm_2015010100_c00.grib2


...

Downloading: https://noaa-gefs-retrospective.s3.amazonaws.com/GEFSv12/reforecast/2015/2015010200/c00/Days:1-10/apcp_sfc_2015010200_c00.grib2


Ignoring index file 'gefs_downloads\\2015\\apcp_sfc\\apcp_sfc_2015010200_c00.grib2.5b7b6.idx' older than GRIB file


Downloading: https://noaa-gefs-retrospective.s3.amazonaws.com/GEFSv12/reforecast/2015/2015010200/c00/Days:1-10/tcdc_eatm_2015010200_c00.grib2


Ignoring index file 'gefs_downloads\\2015\\tcdc_eatm\\tcdc_eatm_2015010200_c00.grib2.5b7b6.idx' older than GRIB file


        time          valid_time  lead_hours  latitude  longitude  \
0 2015-01-01 2015-01-02 06:00:00          30      50.0      235.0   
1 2015-01-01 2015-01-02 06:00:00          30      50.0      236.0   
2 2015-01-01 2015-01-02 06:00:00          30      50.0      237.0   
3 2015-01-01 2015-01-02 06:00:00          30      50.0      238.0   
4 2015-01-01 2015-01-02 06:00:00          30      50.0      239.0   

   precipitation  cloud_cover  year         run  predicted_temp_c  
0            0.0         90.0  2015  2015010100          3.480927  
1            0.0         95.0  2015  2015010100         -2.639069  
2            0.0         89.0  2015  2015010100         -5.359070  
3            0.0         81.0  2015  2015010100         -6.209076  
4            0.0         93.0  2015  2015010100         -6.709076  
(6344, 10)


In [ ]:
# ============================================================
# NOAA/NCEP REANALYSIS 2 ACTUAL 2M TEMPERATURE
# ============================================================


START_DATE = "2015-01-01"
END_DATE = "2015-01-02"

url = (
    "https://psl.noaa.gov/thredds/dodsC/"
    "Datasets/ncep.reanalysis2/gaussian_grid/air.2m.gauss.2015.nc"
)

# ------------------------------------------------------------
# Wait before requesting NOAA server
# ------------------------------------------------------------
time.sleep(5)   # wait 5 seconds before opening the dataset

# ------------------------------------------------------------
# Open NOAA/NCEP Reanalysis 2 actual 2m air temperature
# Retry if the server is busy / rate limiting
# ------------------------------------------------------------
max_retries = 1
wait_seconds = 30

for attempt in range(max_retries):
    try:
        print(f"Opening NOAA dataset... attempt {attempt + 1}/{max_retries}")
        ds_actual = xr.open_dataset(url)
        break

    except Exception as e:
        print("Error opening dataset:")
        print(e)

        if attempt < max_retries - 1:
            print(f"Waiting {wait_seconds} seconds before trying again...")
            time.sleep(wait_seconds)
        else:
            raise

# Keep only requested date range
ds_actual = ds_actual.sel(time=slice(START_DATE, END_DATE))

# USA approximate box.
# Reanalysis latitude runs north -> south, so slice(50, 25) is correct.
# Longitude uses 0-360 degrees, so 235-295 is western/central USA.
ds_actual = ds_actual.sel(
    lat=slice(50, 25),
    lon=slice(235, 295)
)

# Optional pause before pulling data into dataframe
time.sleep(5)

# Convert xarray to dataframe
actual_df = ds_actual["air"].to_dataframe().reset_index()

# Convert actual temperature from Kelvin to Celsius
actual_df["actual_temp_2m_c"] = actual_df["air"] - 273.15

# Keep only useful columns
actual_df = actual_df[["time", "lat", "lon", "actual_temp_2m_c"]]

# Make sure time is datetime
actual_df["time"] = pd.to_datetime(actual_df["time"])

print(actual_df.head())
print(actual_df.shape)

Opening NOAA dataset... attempt 1/1
Error opening dataset:
[Errno -70] NetCDF: DAP server error: 'https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis2/gaussian_grid/air.2m.gauss.2015.nc'


OSError: [Errno -70] NetCDF: DAP server error: 'https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis2/gaussian_grid/air.2m.gauss.2015.nc'

In [ ]:
# ============================================================
# MATCH ACTUAL TEMPERATURE TO EACH FORECAST ROW
#
# forecast_df and actual_df are on different grids.
# Therefore we do NOT merge by exact lat/lon.
# Instead, for each forecast row, we find the nearest actual
# temperature using:
#   forecast valid_time -> actual time
#   forecast latitude   -> actual lat
#   forecast longitude  -> actual lon
#
# final_df will keep the same number of rows as forecast_df.
# ============================================================

# Safety checks so errors are clearer
required_forecast_cols = {"valid_time", "latitude", "longitude", "predicted_temp_c"}
required_actual_cols = {"time", "lat", "lon", "actual_temp_2m_c"}

missing_forecast = required_forecast_cols - set(forecast_df.columns)
missing_actual = required_actual_cols - set(actual_df.columns)

if missing_forecast:
    raise ValueError(f"forecast_df is missing columns: {missing_forecast}")

if missing_actual:
    raise ValueError(f"actual_df is missing columns: {missing_actual}")

# Make copies so the original dataframes stay unchanged
final_df = forecast_df.copy()
actual_lookup_df = actual_df.copy()

# Make sure time columns are datetime
final_df["valid_time"] = pd.to_datetime(final_df["valid_time"])
actual_lookup_df["time"] = pd.to_datetime(actual_lookup_df["time"])

# Convert actual dataframe into an xarray grid for nearest-neighbour lookup
actual_grid = (
    actual_lookup_df
    .set_index(["time", "lat", "lon"])
    ["actual_temp_2m_c"]
    .to_xarray()
)

# Function used for each forecast row
def get_nearest_actual_temp(row):
    """
    Finds the nearest actual/reanalysis 2m temperature for one forecast row.
    """
    return float(
        actual_grid.sel(
            time=row["valid_time"],
            lat=row["latitude"],
            lon=row["longitude"],
            method="nearest"
        ).values
    )

# Add actual temperature to every forecast row
final_df["actual_temp_2m_c"] = final_df.apply(get_nearest_actual_temp, axis=1)

# Calculate model error
# Positive error means the forecast was too cold.
# Negative error means the forecast was too warm.
final_df["model_error_c"] = (
    final_df["actual_temp_2m_c"] - final_df["predicted_temp_c"]
)

print(final_df.head())
print(final_df.shape)

# Optional: save the matched dataset
final_df.to_csv("gefs_forecast_with_actual_temp.csv", index=False)
print("Saved: gefs_forecast_with_actual_temp.csv")



In [ ]:
# ============================================================
# ADD COPERNICUS LAND COVER + 20-DEGREE LOCATION BLOCKS + ENCODING
#
# This cell starts from final_df, because final_df already contains:
#   - predicted_temp_c
#   - actual_temp_2m_c
#   - model_error_c
#
# It then creates:
#   - longitude_180
#   - land_code
#   - land_type
#   - land_use_type
#   - lat_block / lon_block / location_block
#   - encoded_df, ready for machine learning
# ============================================================



# ------------------------------------------------------------
# 0. Start from final_df, NOT the old forecast_df
# ------------------------------------------------------------
# final_df was created in the previous cell after matching actual
# temperature and calculating model_error_c.

if "final_df" not in globals():
    raise NameError("Run the previous cell first. final_df does not exist yet.")

forecast_df = final_df.copy()


# ------------------------------------------------------------
# 1. Path to your downloaded Copernicus GeoTIFF
# ------------------------------------------------------------

landcover_tif = Path(
    "PROBAV_LC100_global_v3.0.1_2015-base_Discrete-Classification-map_EPSG-4326.tif"
)

if not landcover_tif.exists():
    raise FileNotFoundError(
        f"Could not find {landcover_tif}\n"
        "Put the Copernicus .tif file in the same folder as this notebook."
    )


# ------------------------------------------------------------
# 2. Convert NOAA longitude from 0-360 to -180 to 180
# ------------------------------------------------------------
# NOAA example:
#   235 = -125
#   260 = -100
#   295 = -65
# Copernicus/rasterio expects normal longitude values: -180 to 180.

forecast_df["longitude_180"] = forecast_df["longitude"].apply(
    lambda x: x - 360 if x > 180 else x
)


# ------------------------------------------------------------
# 3. Sample Copernicus land-cover code at each lat/lon point
# ------------------------------------------------------------
# rasterio wants coordinates as:
#   (longitude, latitude)

coords = list(zip(
    forecast_df["longitude_180"],
    forecast_df["latitude"]
))

land_codes = []

with rasterio.open(landcover_tif) as src:
    for value in src.sample(coords):
        code = int(value[0])

        # Some rasters use 255 for missing/unknown.
        if code in [255]:
            land_codes.append(np.nan)
        else:
            land_codes.append(code)

forecast_df["land_code"] = land_codes


# ------------------------------------------------------------
# 4. Convert Copernicus numeric codes into land type names
# ------------------------------------------------------------

copernicus_land_classes = {
    0: "Unknown",
    20: "Shrubs",
    30: "Herbaceous vegetation",
    40: "Cultivated and managed vegetation / agriculture",
    50: "Urban / built up",
    60: "Bare / sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    100: "Moss and lichen",

    111: "Closed forest, evergreen needle leaf",
    112: "Closed forest, evergreen broad leaf",
    113: "Closed forest, deciduous needle leaf",
    114: "Closed forest, deciduous broad leaf",
    115: "Closed forest, mixed",
    116: "Closed forest, unknown",

    121: "Open forest, evergreen needle leaf",
    122: "Open forest, evergreen broad leaf",
    123: "Open forest, deciduous needle leaf",
    124: "Open forest, deciduous broad leaf",
    125: "Open forest, mixed",
    126: "Open forest, unknown",

    200: "Open sea"
}

forecast_df["land_type"] = forecast_df["land_code"].map(copernicus_land_classes)
forecast_df["land_type"] = forecast_df["land_type"].fillna("Unknown")


# ------------------------------------------------------------
# 5. Create broader land-use type groups
# ------------------------------------------------------------

def classify_land_use_type(land_type):
    if pd.isna(land_type):
        return "Unknown"

    land_type_lower = str(land_type).lower()

    if "urban" in land_type_lower or "built" in land_type_lower:
        return "Urban"

    elif "cultivated" in land_type_lower or "agriculture" in land_type_lower:
        return "Agriculture"

    elif "forest" in land_type_lower:
        return "Forest"

    elif "water" in land_type_lower or "sea" in land_type_lower:
        return "Water"

    elif "wetland" in land_type_lower:
        return "Wetland"

    elif "snow" in land_type_lower or "ice" in land_type_lower:
        return "Snow/Ice"

    elif (
        "shrub" in land_type_lower
        or "herbaceous" in land_type_lower
        or "moss" in land_type_lower
        or "lichen" in land_type_lower
    ):
        return "Natural vegetation"

    elif "bare" in land_type_lower or "sparse" in land_type_lower:
        return "Bare ground"

    elif "unknown" in land_type_lower:
        return "Unknown"

    else:
        return "Other"


forecast_df["land_use_type"] = forecast_df["land_type"].apply(classify_land_use_type)


# ------------------------------------------------------------
# 6. Create 20-degree latitude/longitude blocks
# ------------------------------------------------------------
# Latitude normally runs from -90 to 90.
# Adding 90 shifts it to 0 to 180 before dividing into 20-degree blocks.
#
# Longitude here uses NOAA's 0 to 360 format.
# If you prefer -180 to 180 blocks, use longitude_180 instead.

forecast_df["lat_block"] = np.floor((forecast_df["latitude"] + 90) / 20).astype(int)
forecast_df["lon_block"] = np.floor(forecast_df["longitude"] / 20).astype(int)

# Combined location zone. Example: lat block 7 and lon block 11 becomes "7_11".
forecast_df["location_block"] = (
    forecast_df["lat_block"].astype(str)
    + "_"
    + forecast_df["lon_block"].astype(str)
)


# ------------------------------------------------------------
# 7. Check the unencoded result first
# ------------------------------------------------------------

print("Unencoded dataframe preview:")
print(
    forecast_df[
        [
            "valid_time",
            "latitude",
            "longitude",
            "longitude_180",
            "predicted_temp_c",
            "actual_temp_2m_c",
            "model_error_c",
            "land_code",
            "land_type",
            "land_use_type",
            "lat_block",
            "lon_block",
            "location_block"
        ]
    ].head(20)
)

print("forecast_df shape:", forecast_df.shape)


# ------------------------------------------------------------
# 8. One-hot encode categorical columns
# ------------------------------------------------------------
# Keep forecast_df unencoded for checking.
# Create encoded_df for machine learning.
#
# We encode location_block instead of separately encoding lat_block
# and lon_block, because location_block already combines both.

categorical_cols = [
    "land_type",
    "land_use_type",
    "location_block"
]

encoded_df = pd.get_dummies(
    forecast_df,
    columns=categorical_cols,
    dtype=int
)

print("encoded_df shape:", encoded_df.shape)
print("Encoded columns added:")
print([c for c in encoded_df.columns if c.startswith("land_type_")][:10])
print([c for c in encoded_df.columns if c.startswith("land_use_type_")][:10])
print([c for c in encoded_df.columns if c.startswith("location_block_")][:10])

encoded_df.columns

In [ ]:
# ============================================================
# MODEL 1: Five-base-model stacking ensemble
#
# Target:
#   model_error_c = actual_temp_2m_c - predicted_temp_c
#
# Base models:
#   1. Ridge
#   2. Random Forest
#   3. XGBoost
#   4. LightGBM
#   5. GBDT
#
# Meta model / stacking layer:
#   Linear Regression
# ============================================================


# ------------------------------------------------------------
# 1. Start with encoded data
# ------------------------------------------------------------


df = encoded_df.copy()

print("Encoded data shape:", df.shape)
print("Columns:", list(df.columns))


# ------------------------------------------------------------
# 2. Remove rows with missing target
# ------------------------------------------------------------

df = df.dropna(subset=["model_error_c"])


# ------------------------------------------------------------
# 3. Define target
# ------------------------------------------------------------
# We are predicting forecast error:
#   model_error_c = actual_temp_2m_c - predicted_temp_c
#
# If model_error_c is positive, the original forecast was too cold.
# If model_error_c is negative, the original forecast was too warm.
# ------------------------------------------------------------

y = df["model_error_c"]


# ------------------------------------------------------------
# 4. Define input features
# ------------------------------------------------------------
# Do NOT include actual_temp_2m_c or model_error_c in X.
# That would leak the answer into the model.
# ------------------------------------------------------------

drop_cols = [
    "model_error_c",
    "actual_temp_2m_c",
    "valid_time",
    "forecast_start_time",
    "longitude_180",
    "land_code"
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Keep only numeric / boolean columns.
# One-hot encoded columns are usually numeric or boolean.
X = X.select_dtypes(include=[np.number, "bool"])

print("X shape:", X.shape)
print("y shape:", y.shape)


# ------------------------------------------------------------
# 5. Train / validation / test split
# ------------------------------------------------------------
# Paper split:
#   70% training
#   20% validation
#   10% test
#
# Training set:
#   used to train the five base models.
#
# Validation set:
#   used to train the Linear Regression stacking model.
#
# Test set:
#   held back for final evaluation.
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1/3,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# ------------------------------------------------------------
# 6. Metric function
# ------------------------------------------------------------

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    # Avoid divide-by-zero problems in MAPE.
    mask = actual != 0
    if mask.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }


# ------------------------------------------------------------
# 7. Define the five base models
# ------------------------------------------------------------

base_models = {
    "ridge": Ridge(alpha=1.0),

    "rf": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "gbdt": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "xgb" = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    "lgbm" = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="regression",
        random_state=42,
        n_jobs=-1
    )
}

# ------------------------------------------------------------
# 8. Train base models and collect their predictions
# ------------------------------------------------------------

trained_base_models = {}

base_train_predictions = pd.DataFrame(index=X_train.index)
base_val_predictions = pd.DataFrame(index=X_val.index)
base_test_predictions = pd.DataFrame(index=X_test.index)

base_model_results = []

for name, model in base_models.items():
    print("Training base model:", name)

    model.fit(X_train, y_train)
    trained_base_models[name] = model

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    base_train_predictions[f"{name}_pred"] = train_pred
    base_val_predictions[f"{name}_pred"] = val_pred
    base_test_predictions[f"{name}_pred"] = test_pred

    for dataset_name, y_split, preds in [
        ("Training Set", y_train, train_pred),
        ("Validation Set", y_val, val_pred),
        ("Test Set", y_test, test_pred)
    ]:
        base_model_results.append({
            "Model": name,
            "Dataset": dataset_name,
            **calculate_metrics(y_split, preds)
        })

base_results_df = pd.DataFrame(base_model_results)


# ------------------------------------------------------------
# 9. Train stacking model
# ------------------------------------------------------------
# The stacker does not use the original columns directly.
# It only sees the five base-model predictions:
#
#   ridge_pred, rf_pred, xgb_pred, lgbm_pred, gbdt_pred
# ------------------------------------------------------------

ensemble_features = list(base_val_predictions.columns)

X_stack_train = base_val_predictions[ensemble_features]
y_stack_train = y_val

stacking_model = LinearRegression()
stacking_model.fit(X_stack_train, y_stack_train)

print("Stacking features:", ensemble_features)
print("Stacking coefficients:")
for feature, coef in zip(ensemble_features, stacking_model.coef_):
    print(f"  {feature}: {coef:.6f}")
print("Stacking intercept:", stacking_model.intercept_)


# ------------------------------------------------------------
# 10. Final stacked predictions
# ------------------------------------------------------------

stack_train_pred = stacking_model.predict(base_train_predictions[ensemble_features])
stack_val_pred = stacking_model.predict(base_val_predictions[ensemble_features])
stack_test_pred = stacking_model.predict(base_test_predictions[ensemble_features])

stack_results_df = pd.DataFrame([
    {
        "Model": "Stacked_LinearRegression",
        "Dataset": "Training Set",
        **calculate_metrics(y_train, stack_train_pred)
    },
    {
        "Model": "Stacked_LinearRegression",
        "Dataset": "Validation Set",
        **calculate_metrics(y_val, stack_val_pred)
    },
    {
        "Model": "Stacked_LinearRegression",
        "Dataset": "Test Set",
        **calculate_metrics(y_test, stack_test_pred)
    }
])

results_df = pd.concat([base_results_df, stack_results_df], ignore_index=True)


# ------------------------------------------------------------
# 11. Store test-set error predictions for the next cell
# ------------------------------------------------------------

predicted_error_test = stack_test_pred

# Keep base model predictions too, so you can inspect the ensemble.
stack_test_details = base_test_predictions.copy()
stack_test_details["stacked_predicted_error_c"] = predicted_error_test
stack_test_details["actual_model_error_c"] = y_test.values

print("\nModel results:")
print(results_df.sort_values(["Dataset", "RMSE"]))

results_df


In [ ]:
# ============================================================
# Compare original forecast vs stacked corrected forecast
# ============================================================


# ------------------------------------------------------------
# 1. Get original rows for the held-out test set
# ------------------------------------------------------------

test_output = df.loc[X_test.index].copy()


# ------------------------------------------------------------
# 2. Store stacked predicted error
# ------------------------------------------------------------

test_output["predicted_error_c"] = predicted_error_test


# ------------------------------------------------------------
# 3. Correct the original forecast
# ------------------------------------------------------------
# model_error_c = actual_temp_2m_c - predicted_temp_c
# therefore:
# corrected_temp_c = predicted_temp_c + predicted_error_c
# ------------------------------------------------------------

test_output["corrected_temp_c"] = (
    test_output["predicted_temp_c"] + test_output["predicted_error_c"]
)


# ------------------------------------------------------------
# 4. Metric function
# ------------------------------------------------------------

def calculate_forecast_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    mask = actual != 0
    if mask.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }


# ------------------------------------------------------------
# 5. Original forecast vs corrected forecast
# ------------------------------------------------------------

original_metrics = calculate_forecast_metrics(
    test_output["actual_temp_2m_c"],
    test_output["predicted_temp_c"]
)

corrected_metrics = calculate_forecast_metrics(
    test_output["actual_temp_2m_c"],
    test_output["corrected_temp_c"]
)

comparison_df = pd.DataFrame([
    {
        "Forecast Type": "Original Forecast",
        **original_metrics
    },
    {
        "Forecast Type": "Stacked Corrected Forecast",
        **corrected_metrics
    }
])

print(comparison_df)

if corrected_metrics["RMSE"] < original_metrics["RMSE"]:
    print("Good: stacked correction improved the forecast RMSE.")
else:
    print("Warning: stacked correction did not improve the forecast RMSE yet.")


# ------------------------------------------------------------
# 6. Add individual base-model error predictions for inspection
# ------------------------------------------------------------

if "base_test_predictions" in globals():
    for col in base_test_predictions.columns:
        test_output[col] = base_test_predictions[col]


# ------------------------------------------------------------
# 7. Show sample results safely
# ------------------------------------------------------------

wanted_cols = [
    "latitude",
    "longitude",
    "lead_hours",
    "actual_temp_2m_c",
    "predicted_temp_c",
    "ridge_pred",
    "rf_pred",
    "xgb_pred",
    "lgbm_pred",
    "gbdt_pred",
    "predicted_error_c",
    "corrected_temp_c"
]

existing_cols = [c for c in wanted_cols if c in test_output.columns]

test_output[existing_cols].head(20)
